# Training Bottleneck Analysis (PyTorch)

This notebook profiles where time is spent during training and compares `num_workers` settings for your dataset and model stack.

Sections:
1. Environment summary
2. DataLoader-only benchmark by `num_workers`
3. Per-step timing breakdown (`data wait`, `to device`, `forward`, `loss/backward`, `optimizer`)
4. `torch.profiler` operator table
5. Practical recommendations from measured results

In [1]:
import os
import time
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torch.profiler import profile, ProfilerActivity

from lib.ensemble import SegmentationEnsemble

pd.set_option('display.max_columns', 100)

if torch.cuda.is_available():
    active_device = 'cuda'
elif torch.backends.mps.is_available():
    active_device = 'mps'
else:
    active_device = 'cpu'

print('OS:', platform.platform())
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('Active device:', active_device)

OS: macOS-26.3.1-arm64-arm-64bit
Python: 3.12.9
PyTorch: 2.9.1
Active device: mps


In [2]:
# Pick one training config and one set-model for profiling.
CONFIG_PATH = 'configs/iou_full.toml'
SET_INDEX = 0
MAX_BATCHES_BENCH = 30

ensemble = SegmentationEnsemble(CONFIG_PATH)
model_runner = ensemble.models[SET_INDEX]

print('Config:', CONFIG_PATH)
print('Set:', model_runner.set_id)
print('Train samples:', len(model_runner.train_dataset))
print('Batch size:', model_runner.train_loader.batch_size)
print('Default num_workers in loader:', model_runner.train_loader.num_workers)

Using device: mps
Channels per layer: [16, 32, 64, 128, 256, 256]
Model initialized with 6.89M parameters.
Using device: mps
Channels per layer: [16, 32, 64, 128, 256, 256]
Model initialized with 6.89M parameters.
Using device: mps
Channels per layer: [16, 32, 64, 128, 256, 256]
Model initialized with 6.89M parameters.
Using device: mps
Channels per layer: [16, 32, 64, 128, 256, 256]
Model initialized with 6.89M parameters.
Config: configs/iou_full.toml
Set: 0
Train samples: 14
Batch size: 8
Default num_workers in loader: 0


In [7]:
def benchmark_dataloader(dataset, batch_size, worker_values, max_batches=30, persistent=False):
    rows = []
    for nw in worker_values:
        kwargs = dict(
            dataset=dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=int(nw),
            drop_last=False,
        )
        if nw > 0:
            kwargs['persistent_workers'] = bool(persistent)
            kwargs['prefetch_factor'] = 2

        loader = DataLoader(**kwargs)

        wait_times = []
        t_iter_start = time.perf_counter()
        it = iter(loader)

        for _ in range(max_batches):
            t0 = time.perf_counter()
            try:
                _ = next(it)
            except StopIteration:
                break
            t1 = time.perf_counter()
            wait_times.append(t1 - t0)

        total_iter_time = time.perf_counter() - t_iter_start
        n = len(wait_times)

        rows.append({
            'num_workers': nw,
            'persistent_workers': bool(persistent) if nw > 0 else False,
            'batches': n,
            'mean_batch_wait_s': float(np.mean(wait_times)) if n else np.nan,
            'p95_batch_wait_s': float(np.percentile(wait_times, 95)) if n else np.nan,
            'epoch_like_iter_s': float(total_iter_time),
            'batches_per_s': float(n / total_iter_time) if total_iter_time > 0 and n > 0 else np.nan,
        })

    return pd.DataFrame(rows)

worker_values = [0, 1, 2, 4]
dl_bench_nonpersistent = benchmark_dataloader(
    model_runner.train_dataset,
    model_runner.train_loader.batch_size,
    worker_values=worker_values,
    max_batches=MAX_BATCHES_BENCH,
    persistent=False,
)

dl_bench_persistent = benchmark_dataloader(
    model_runner.train_dataset,
    model_runner.train_loader.batch_size,
    worker_values=[w for w in worker_values if w > 0],
    max_batches=MAX_BATCHES_BENCH,
    persistent=True,
)

display(pd.concat([dl_bench_nonpersistent, dl_bench_persistent], ignore_index=True).round(4))

,num_workers,persistent_workers,batches,mean_batch_wait_s,p95_batch_wait_s,epoch_like_iter_s,batches_per_s
0,0,False,2,0.0613,0.0759,0.1234,16.2042
1,1,False,2,0.7035,1.3007,6.4232,0.3114
2,2,False,2,0.6502,1.2354,11.3197,0.1767
3,4,False,2,0.8729,1.6584,11.7845,0.1697
4,1,True,2,0.7083,1.3119,1.4214,1.4071
5,2,True,2,0.0006,0.0011,5.0124,0.3990
6,4,True,2,0.0005,0.0007,10.0228,0.1995


In [8]:
def move_batch_to_device(batch, device):
    return {
        'image': batch['image'].to(device),
        'label': batch['label'].to(device),
        'edge_labels': batch['edge_labels'].to(device),
    }

def sync_if_needed(device_type):
    if device_type == 'cuda' and torch.cuda.is_available():
        torch.cuda.synchronize()
    elif device_type == 'mps' and torch.backends.mps.is_available():
        torch.mps.synchronize()

def benchmark_train_step_breakdown(model_runner, num_workers=0, steps=20):
    device = model_runner.device
    model = model_runner.model
    criterion = model_runner.train_criterion
    optimizer = model_runner.optimizer

    loader = DataLoader(
        model_runner.train_dataset,
        batch_size=model_runner.train_loader.batch_size,
        shuffle=False,
        num_workers=num_workers,
        drop_last=False,
        persistent_workers=(num_workers > 0),
    )

    model.train()
    it = iter(loader)
    rows = []

    for step in range(steps):
        t0 = time.perf_counter()
        try:
            batch = next(it)
        except StopIteration:
            break
        t1 = time.perf_counter()

        batch_dev = move_batch_to_device(batch, device)
        sync_if_needed(device.type)
        t2 = time.perf_counter()

        optimizer.zero_grad(set_to_none=True)
        outputs = model(batch_dev['image'])
        sync_if_needed(device.type)
        t3 = time.perf_counter()

        loss = criterion(outputs, batch_dev['label'])
        loss.backward()
        sync_if_needed(device.type)
        t4 = time.perf_counter()

        optimizer.step()
        sync_if_needed(device.type)
        t5 = time.perf_counter()

        rows.append({
            'step': step,
            'data_wait_s': t1 - t0,
            'to_device_s': t2 - t1,
            'forward_s': t3 - t2,
            'loss_backward_s': t4 - t3,
            'optimizer_s': t5 - t4,
            'total_step_s': t5 - t0,
            'loss': float(loss.detach().cpu().item()),
        })

    return pd.DataFrame(rows)

step_breakdown_nw0 = benchmark_train_step_breakdown(model_runner, num_workers=0, steps=20)
step_breakdown_nw2 = benchmark_train_step_breakdown(model_runner, num_workers=2, steps=20)

summary = []
for name, df in [('nw=0', step_breakdown_nw0), ('nw=2', step_breakdown_nw2)]:
    if len(df) == 0:
        continue
    means = df[['data_wait_s', 'to_device_s', 'forward_s', 'loss_backward_s', 'optimizer_s', 'total_step_s']].mean()
    shares = means[['data_wait_s', 'to_device_s', 'forward_s', 'loss_backward_s', 'optimizer_s']] / max(means['total_step_s'], 1e-12)
    row = {'setting': name, **{k: float(v) for k, v in means.items()}}
    row.update({f'{k}_share': float(v) for k, v in shares.items()})
    summary.append(row)

step_summary = pd.DataFrame(summary)
display(step_summary.round(4))

,setting,data_wait_s,to_device_s,forward_s,loss_backward_s,optimizer_s,total_step_s,data_wait_s_share,to_device_s_share,forward_s_share,loss_backward_s_share,optimizer_s_share
0,nw=0,0.0519,0.0821,0.1485,0.1329,0.0176,0.4330,0.1199,0.1896,0.3429,0.3070,0.0405
1,nw=2,0.7269,0.0139,0.1055,0.1838,0.0245,1.0546,0.6893,0.0132,0.1000,0.1743,0.0232


In [11]:
# Torch profiler on a few steps (operator-level).
activities = [ProfilerActivity.CPU]
if model_runner.device.type == 'cuda':
    activities.append(ProfilerActivity.CUDA)

loader = DataLoader(
    model_runner.train_dataset,
    batch_size=model_runner.train_loader.batch_size,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)

it = iter(loader)
model_runner.model.train()

with profile(activities=activities, record_shapes=False, with_stack=False) as prof:
    for _ in range(8):
        try:
            batch = next(it)
        except StopIteration:
            break

        batch_dev = {
            'image': batch['image'].to(model_runner.device),
            'label': batch['label'].to(model_runner.device),
        }

        model_runner.optimizer.zero_grad(set_to_none=True)
        out = model_runner.model(batch_dev['image'])
        loss = model_runner.train_criterion(out, batch_dev['label'])
        loss.backward()
        model_runner.optimizer.step()

sort_key = 'self_cpu_time_total'
print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                         aten::addcmul_        43.66%     397.089ms        43.66%     397.089ms       1.838ms           216  
                                            aten::copy_        30.76%     279.789ms        30.77%     279.810ms     274.863us          1018  
enumerate(DataLoader)#_SingleProcessDataLoaderIter._...        12.09%     109.954ms        12.89%     117.199ms      39.066ms             3  
                         aten::mps_convolution_backward         3.45%      31.333ms         3.46%      31.480ms     524.673us            60  
      

In [10]:
def summarize_findings(dl_nonpersistent, step_summary_df):
    print('=== Quick Findings ===')
    if len(dl_nonpersistent) > 0:
        best = dl_nonpersistent.sort_values('batches_per_s', ascending=False).iloc[0]
        worst = dl_nonpersistent.sort_values('batches_per_s', ascending=True).iloc[0]
        print(f"Best DataLoader throughput: nw={int(best['num_workers'])}, {best['batches_per_s']:.3f} batches/s")
        print(f"Worst DataLoader throughput: nw={int(worst['num_workers'])}, {worst['batches_per_s']:.3f} batches/s")

    if len(step_summary_df) > 0:
        for _, r in step_summary_df.iterrows():
            print(f"\nSetting {r['setting']}:")
            print(f"- total_step_s: {r['total_step_s']:.4f}")
            print(f"- data_wait share: {100*r['data_wait_s_share']:.1f}%")
            print(f"- forward share: {100*r['forward_s_share']:.1f}%")
            print(f"- backward share: {100*r['loss_backward_s_share']:.1f}%")

    print('\n=== Actionable Next Tweaks to Test ===')
    print('1. Keep num_workers at the fastest measured value (often 0 or 1 on macOS/MPS).')
    print('2. If using workers > 0, test persistent_workers=True and prefetch_factor=2.')
    print('3. Disable expensive augmentations/edge maps temporarily to isolate preprocessing cost.')
    print('4. Reduce synchronization points in non-profiling training code (synchronize only for benchmarking).')

summarize_findings(dl_bench_nonpersistent, step_summary)

=== Quick Findings ===
Best DataLoader throughput: nw=0, 16.204 batches/s
Worst DataLoader throughput: nw=4, 0.170 batches/s

Setting nw=0:
- total_step_s: 0.4330
- data_wait share: 12.0%
- forward share: 34.3%
- backward share: 30.7%

Setting nw=2:
- total_step_s: 1.0546
- data_wait share: 68.9%
- forward share: 10.0%
- backward share: 17.4%

=== Actionable Next Tweaks to Test ===
1. Keep num_workers at the fastest measured value (often 0 or 1 on macOS/MPS).
2. If using workers > 0, test persistent_workers=True and prefetch_factor=2.
3. Disable expensive augmentations/edge maps temporarily to isolate preprocessing cost.
4. Reduce synchronization points in non-profiling training code (synchronize only for benchmarking).
